<a href="https://colab.research.google.com/github/agmCorp/colab/blob/main/matching_catalogos_bse_autodata_v6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema de Matching Optimizado v6: BSE vs Autodata

## ¿Qué hace este notebook?

Resuelve el problema de **vincular automáticamente dos catálogos de vehículos** que describen los mismos modelos con nomenclaturas distintas: el catálogo **BSE** (aprox. 33k registros) y el catálogo **Autodata** (aprox. 20k registros). Para cada registro de Autodata se encuentra el mejor equivalente en BSE, junto con un score de confianza [0, 1].

### Proceso general

1. **Preprocesamiento**: Ambos catálogos se normalizan para eliminar inconsistencias tipográficas, caracteres corruptos y variaciones de escritura. Se aplica un diccionario de sinónimos del dominio automotor (ej: `AA = A/A = aire acondicionado`, `TDI = TD Intercooler`, `5p = 5 puertas`, etc.) para que términos equivalentes queden representados con el mismo token antes de comparar. Los años erróneos del catálogo BSE (ej: `3020 → 2020`) también se corrigen en esta etapa.

2. **Índices de búsqueda**: Se construyen tres índices sobre BSE para acelerar el matching: un índice por marca+año (filtrado rápido), embeddings vectoriales con el modelo BGE-M3 (similitud semántica) e índices BM25 por marca (similitud léxica).

3. **Matching por registro**: Para cada registro Autodata se ejecuta un pipeline de tres fases:
   - **Pre-filtrado**: se limita el espacio de búsqueda a registros BSE de la misma marca cuyo rango de años incluye exactamente el año del registro Autodata (`anio_inicio_bse ≤ ANIO_AUTODATA ≤ anio_fin_bse`). Para registros con año desconocido (`9999`), se consideran todos los registros de la marca.
   - **Búsqueda híbrida**: se combinan similitud semántica (BGE-M3) y léxica (BM25) para obtener los 50 candidatos más prometedores.
   - **Reranking**: un cross-encoder (BGE-Reranker-v2-m3) decide el orden final. El score resultante se convierte a [0,1] con una función sigmoid de temperatura calibrada automáticamente. Para registros con año desconocido (`9999`) se aplica adicionalmente una penalización leve de `0.9` sobre el score final.

4. **Salida**: Para cada registro Autodata se generan hasta 3 filas en el archivo completo —una por cada candidato BSE encontrado—, repitiendo el `ID_AUTODATA` en cada fila. Cada fila incluye: `RANK` (posición 1-3), score, marca, modelo, rango de años, combustible y tipo. El archivo resumido conserva únicamente la fila `RANK == 1` (mejor match) por registro.


## 1️⃣ Instalación de dependencias

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Instalar librerías necesarias
!pip install -q sentence-transformers pandas numpy unidecode tqdm rank_bm25 openpyxl

## 2️⃣ Importar librerías

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from unidecode import unidecode
import re
from tqdm.auto import tqdm
import warnings
import torch
from collections import defaultdict
import os

warnings.filterwarnings('ignore')
print("Librerías importadas correctamente")
print(f"GPU disponible: {torch.cuda.is_available()}")

## 3️⃣ Cargar los archivos CSV

In [ ]:
from google.colab import drive

# Montar Google Drive (abre un popup para autorizar el acceso la primera vez)
drive.mount('/content/drive')

# Rutas configurables
DRIVE_INPUT_PATH  = '/content/drive/MyDrive/Colab Notebooks/input'
DRIVE_OUTPUT_PATH = '/content/drive/MyDrive/Colab Notebooks/output'

# Crear carpeta de salida si no existe
os.makedirs(DRIVE_OUTPUT_PATH, exist_ok=True)

# Rutas de los archivos de entrada
bse_path      = os.path.join(DRIVE_INPUT_PATH, 'catalogo-bse.xlsx')
autodata_path = os.path.join(DRIVE_INPUT_PATH, 'catalogo-autodata.xlsx')

print(f'Input:  {DRIVE_INPUT_PATH}')
print(f'Output: {DRIVE_OUTPUT_PATH}')

# Cargar Excel (todo como string)
df_bse      = pd.read_excel(bse_path,      dtype=str)
df_autodata = pd.read_excel(autodata_path, dtype=str)

# Reemplazar NaN por string vacío
df_bse      = df_bse.fillna('')
df_autodata = df_autodata.fillna('')

# Eliminar espacios al inicio y fin de los textos
df_bse      = df_bse.apply(lambda col: col.str.strip())
df_autodata = df_autodata.apply(lambda col: col.str.strip())

# Verificación rápida
print('BSE shape:', df_bse.shape)
print('AUTODATA shape:', df_autodata.shape)

print('\nColumnas BSE:')
print(df_bse.columns)

print('\nColumnas AUTODATA:')
print(df_autodata.columns)

# Mostrar primeras filas
display(df_bse.head(15))
display(df_autodata.head(15))


## 4️⃣ Funciones de Preprocesamiento

In [ ]:

# Detecta caracteres fuera del rango ASCII estándar para anticipar problemas de encoding
def detectar_caracteres_especiales(df, columna):
    todos_los_textos = " ".join(df[columna].astype(str).unique())
    caracteres_raros = set(re.findall(r'[^a-zA-Z0-9\s.,\-\/]', todos_los_textos))
    return caracteres_raros

def estandarizar_terminos_tecnicos(texto):
    """Normaliza sinónimos técnicos a tokens estándar únicos."""
    if not texto: return ""
    # normalizar_texto aplica .lower() antes de invocar esta función
    texto_procesado = texto

    # c/ → "con" solo cuando lo que sigue es una letra, no un dígito.
    # Esto evita alterar códigos de modelo con barra como "C/200" o "C/220".
    texto_procesado = re.sub(r'\bc\s*/\s*(?=[a-z])', ' con ', texto_procesado)
    texto_procesado = re.sub(r'\bp\s*/\s*', ' para ', texto_procesado)

    mapeos = {
        # AA, A/A, Aire y C/Aire son variantes del mismo equipamiento.
        # "C/Aire" llega aquí ya transformado en "con aire" por la expansión de c/ anterior.
        r"\baire\s+acondicionado\b|\ba/a\b|\baa\b|\baire\b|\bcon aire\b": " aire_acondicionado ",
        r"\bclimatizador\b|\bclim\b|\bclimaut\b": " climatizador ",
        r"\bpas\b|\bpax\b|\bpasajeros\b": " pasajeros ",
        r"\bhb\b|\bhatchback\b": " hatchback ",
        r"\bsb\b|\bsportback\b": " sportback ",
        r"\be\.full\b|\bex\.full\b|\bextra full\b": " extra_full ",
        r"\bs\.full\b|\bsuper full\b": " super_full ",
        r"\bcue\b|\bcuero\b": " cuero ",
        # Tech, Techo y T.Cielo designan el mismo equipamiento según especificación del negocio
        r"\btech\b|\btecho\b|\bt\.cielo\b": " techo_solar ",
        r"\bmb\b|\bmultim\b|\bmultimedia\b": " multimedia ",
        r"\bay\.est\b|\bay\.estac\b|\ba\.estac\b|\bp\.assi\b|\bp\.assist\b|\bpark assist\b": " ayuda_estacionamiento ",
        r"\bllan\b|\bllantas\b": " llantas ",
        # turbo_diesel_intercooler debe ir antes de turbo_diesel para que \btd\b
        # no consuma el "td" dentro de "TD Intercooler" antes de que el patrón compuesto pueda matchearlo
        r"\btdi\b|\btd intercooler\b|\bt\.diesel intercooler\b|\bturbo d intercooler\b": " turbo_diesel_intercooler ",
        r"\btd\b|\bt\.diesel\b|\bturbo d\b": " turbo_diesel ",
        # \baut\. sin \b final: el word boundary falla cuando el punto va seguido de espacio
        # o fin de string, ya que tanto "." como " " son caracteres no-palabra y no existe
        # boundary entre ellos. Tiptronic y S-Tronic se incluyen porque son transmisiones
        # automáticas, aunque algunos registros no lo indiquen explícitamente con "AUT".
        r"\btiptronic\b|\bs-tronic\b|\baut\.|\bautomatico\b": " automatico ",
        r"\b3-5p\b|\b3-5 ptas\b|\b3-5 puertas\b": " 3_5_puertas ",
        r"\b4-5p\b|\b4-5 ptas\b|\b4-5 puertas\b": " 4_5_puertas ",
        r"(?<!-)(?:\b2p\b|\b2 ptas\b|\b2 puertas\b)": " 2_puertas ",
        r"(?<!-)(?:\b3p\b|\b3 ptas\b|\b3 puertas\b)": " 3_puertas ",
        r"(?<!-)(?:\b4p\b|\b4 ptas\b|\b4 puertas\b)": " 4_puertas ",
        r"(?<!-)(?:\b5p\b|\b5 ptas\b|\b5 puertas\b)": " 5_puertas ",
        r"\badaptado a rural\b|\badaptada a rural\b|\bad\.?\s*rural\b": " furgon_reformado ",
        r"\blgo\b|\blargo\b": " largo ",
    }
    for patron, reemplazo in mapeos.items():
        texto_procesado = re.sub(patron, reemplazo, texto_procesado)
    return texto_procesado

def normalizar_texto(texto):
    """Repara caracteres extraños, unifica sinónimos y limpia símbolos."""
    if pd.isna(texto):
        return ""

    texto = str(texto).lower()

    MAPA_REPARACION = {
        "á": "a",
        "é": "e",
        "í": "i",
        "ó": "o",
        "ú": "u",
        "ñ": "n",
        "ã¡": "a",
        "ã©": "e",
        "ã³": "o",
        "ãº": "u",
        "ã±": "n",
        "ã": "i",
        "¿": "o",
        "±": "n",
        "ý": "i",
        "ß": "a",
        "ð": "o",
        "â": "a",
        "²": "2",
        "³": "3",
        "¬": "",
        "·": "",
        "°": "",
        "º": "",
        "*": "",
        "+": "",
        "|": "",
        ")": "",
        '"': "",
        "!": "",
        "]": "",
        "[": "",
        "(": "",
        "?": "",
        "=": "",
    }

    for error, correccion in MAPA_REPARACION.items():
        texto = texto.replace(error, correccion)

    texto = unidecode(texto)
    texto = estandarizar_terminos_tecnicos(texto)

    # Los tokens temporales protegen puntos decimales (ej: "2,0" → "2.0") y guiones
    # internos (ej: "4x4-full") para que no se pierdan en la limpieza posterior de
    # caracteres no alfanuméricos
    decimal_token = "_decimal_tok_"
    hyphen_token  = "_hyphen_tok_"

    texto = re.sub(r"(?<=\d)[\.,](?=\d)", decimal_token, texto)
    texto = re.sub(r'(?<=[a-z0-9])-(?=[a-z0-9])', hyphen_token, texto)
    texto = re.sub(r'(?<=[a-z0-9])/(?=[a-z0-9])', '_', texto)
    texto = re.sub(r'\s+-\s+', ' ', texto)
    texto = re.sub(r"[^a-z0-9_\s]", " ", texto)
    texto = texto.replace(hyphen_token, '-')
    texto = texto.replace(decimal_token, '.')
    return re.sub(r"\s+", " ", texto).strip()

# El catálogo BSE contiene años erróneos en dos rangos: mayores a 3000 (ej: 3020)
# y en el rango 2985-2999 (ej: 2997). Todos corresponden a años reales menos 1000.
# El umbral 2030 cubre ambos casos sin afectar años actuales válidos.
ANIO_MAXIMO_VALIDO = 2030

def corregir_anio_bse(anio):
    """Corrige años erróneos restándoles 1000 (ej: 3020 → 2020, 2997 → 1997)."""
    if anio is None: return None
    return anio - 1000 if anio > ANIO_MAXIMO_VALIDO else anio

def extraer_anio_inicio_fin(rango_anios):
    if pd.isna(rango_anios): return None, None
    rango_str = str(rango_anios).strip()
    match = re.findall(r'(\d{4})', rango_str)
    if len(match) >= 2:
        return corregir_anio_bse(int(match[0])), corregir_anio_bse(int(match[1]))
    elif len(match) == 1:
        anio = corregir_anio_bse(int(match[0]))
        return anio, anio
    return None, None

def normalizar_marca(texto):
    if pd.isna(texto): return ""
    texto = str(texto).lower()
    texto = re.sub(r'<.*?>', '', texto)     # Elimina sufijos de país como <BRA>, <COL>
    texto = unidecode(texto)                # Citroën -> Citroen
    return texto.strip()

# Tabla de equivalencias entre nombres de marca en Autodata y en BSE.
# Necesaria porque algunos fabricantes aparecen con nombres distintos en cada catálogo
# (ej: "kia motors" en Autodata → "kia" en BSE, "dongfeng" → dos variantes en BSE).
ALIAS_MARCAS = {
    "polarsun":         ["polar sun"],
    "routebuggy":       ["route buggy"],
    "zhong tong":       ["zhongtong"],
    "zxauto":           ["zx auto"],
    "tongbao":          ["tong bao"],
    "leapmotor":        ["leap motor"],
    "kia motors":       ["kia"],
    "byd auto":         ["byd"],
    "bjc":              ["bjc-beijing"],
    "gwm":              ["gwm - [great wall motor]"],
    "gac - trumpchi":   ["gac"],
    "lynk co":          ["lynk y co"],
    "porsche replica":  ["porsche"],
    "brilliance sunra": ["brilliance"],
    "shineray":         ["shinerai"],
    "fuqi":             ["fuqui"],
    "dfsk":             ["dfm - dfsk (dong feng)"],
    "chana":            ["chana-changan"],
    "changan":          ["chana-changan"],
    "geely riddara":    ["riddara (geely)"],
    "dongfeng":         ["dfm - dfsk (dong feng)", "forthing (dongfeng)"],
    "daewoo - fso":     ["daewoo", "fso"],
    "vaz 1111 / uaz":   ["vaz", "uaz"],
}

def resolver_marcas_bse(marca_auto_norm):
    """Resuelve una marca normalizada de Autodata a lista de marcas BSE equivalentes."""
    if marca_auto_norm in ALIAS_MARCAS:
        return ALIAS_MARCAS[marca_auto_norm]
    return [marca_auto_norm]

def validar_normalizacion_contextual(df_bse, df_autodata, n_muestra=50):
    """Genera validación rápida con tabla de ejemplos y muestra real para revisión manual."""
    ejemplos = [
        ("c/diesel 2,0",          "con diesel 2.0"),
        ("p/lancha x-line",       "para lancha x-line"),
        ("S-TRONIC 3.3-3.8",      "automatico 3.3-3.8"),
        ("USM-2450-74-ST",        "usm-2450-74-st"),
        ("FULL - EXTRAFULL",      "full extrafull"),
        ("DX/LX 4-5p",            "dx_lx 4_5_puertas"),
        ("Adaptado a rural",       "furgon_reformado"),
        ("ad. rural",              "furgon_reformado"),
        ("AA 1.6",                 "aire_acondicionado 1.6"),
        ("A/A 1.6",                "aire_acondicionado 1.6"),
        ("C/Aire AUT.",            "aire_acondicionado automatico"),
        ("SEDAN AUT. FULL",        "sedan automatico full"),
        ("TECH 5P",                "techo_solar 5_puertas"),
        ("TD Intercooler 4x4",     "turbo_diesel_intercooler 4x4"),
        ("Turbo D Intercooler",    "turbo_diesel_intercooler"),
        ("TDI 2.0",                "turbo_diesel_intercooler 2.0"),
        ("TD 1.9",                 "turbo_diesel 1.9"),
    ]

    df_ejemplos = pd.DataFrame(ejemplos, columns=['entrada', 'salida_esperada'])
    df_ejemplos['salida_obtenida'] = df_ejemplos['entrada'].apply(normalizar_texto)
    df_ejemplos['ok'] = df_ejemplos['salida_obtenida'] == df_ejemplos['salida_esperada']

    serie_real = pd.concat([
        df_bse['MODELO_BSE'].astype(str),
        df_autodata['MODELO_AUTODATA'].astype(str)
    ], ignore_index=True).dropna().drop_duplicates()

    n = min(n_muestra, len(serie_real))
    df_revision = pd.DataFrame({'entrada_real': serie_real.sample(n=n, random_state=42)})
    df_revision['salida_normalizada'] = df_revision['entrada_real'].apply(normalizar_texto)

    return df_ejemplos, df_revision

print("Funciones de preprocesamiento definidas")
print(f"Alias de marcas configurados: {len(ALIAS_MARCAS)} reglas")


## 5️⃣ Preprocesamiento de datos

In [ ]:
print("Iniciando preprocesamiento de catálogos...")

print(
    "Caracteres extraños en BSE:",
    detectar_caracteres_especiales(df_bse, "MODELO_BSE"),
)
print(
    "Caracteres extraños en Autodata:",
    detectar_caracteres_especiales(df_autodata, "MODELO_AUTODATA"),
)

# Excluir registros con ID_BSE vacío o igual a 0, y con COEF AJUSTE distinto de 1
df_bse = df_bse[
    (df_bse["ID_BSE"] != "")
    & (df_bse["ID_BSE"] != "0")
    & (df_bse["COEF_AJUSTE"] == "1")
].copy()

# --- PROCESAR BSE ---
df_bse['marca_norm'] = df_bse['MARCA_BSE'].apply(normalizar_marca)
df_bse['modelo_norm'] = df_bse['MODELO_BSE'].apply(normalizar_texto)
df_bse[["anio_inicio", "anio_fin"]] = df_bse["ANIOS_BSE"].apply(
    lambda x: pd.Series(extraer_anio_inicio_fin(x))
)
# Filtrar registros sin años procesables
df_bse = df_bse[df_bse['anio_inicio'].notna()].copy()
df_bse.reset_index(drop=True, inplace=True)  # IMPORTANTE: resetear índice para indexación consistente con embeddings

# --- PROCESAR AUTODATA ---
df_autodata['marca_norm'] = df_autodata['MARCA_AUTODATA'].apply(normalizar_marca)
df_autodata['modelo_norm'] = df_autodata['MODELO_AUTODATA'].apply(normalizar_texto)
# Año técnico para matching (derivado desde texto)
df_autodata['anio'] = pd.to_numeric(df_autodata['ANIO_AUTODATA'], errors='coerce').fillna(9999).astype(int)

print("Preprocesamiento completado.")
print(f"BSE limpio: {len(df_bse):,} registros | Autodata: {len(df_autodata):,} registros.")

print("\nEjemplo de preprocesamiento:")
print("\nBSE:")
display(df_bse[['MARCA_BSE', 'marca_norm', 'MODELO_BSE', 'modelo_norm', 'ANIOS_BSE', 'anio_inicio', 'anio_fin']].head(10))
print("\nAutodata:")
display(df_autodata[['MARCA_AUTODATA', 'marca_norm', 'MODELO_AUTODATA', 'modelo_norm', 'ANIO_AUTODATA', 'anio']].head(10))

# Validación recomendada antes de ejecutar matching completo
df_validacion_ejemplos, df_revision_50 = validar_normalizacion_contextual(df_bse, df_autodata, n_muestra=50)
print("\nValidación de ejemplos controlados:")
display(df_validacion_ejemplos)
print("\nMuestra de 50 casos reales para revisión manual:")
display(df_revision_50)

## 6️⃣ Cargar Modelos y Construir Índices Optimizados

In [ ]:
# Detectar dispositivo para los modelos de IA (Transformers sí usarán GPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Usando device para IA: {device}")

# ========== 1. MODELOS DE IA ==========
print("\n[1/5] Cargando modelos de embeddings y reranker...")
# BGE-M3 genera los vectores y Reranker decide el mejor candidato
model_bi = SentenceTransformer('BAAI/bge-m3', device=device)
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=device)
print("Modelos cargados")

# ========== 2. ÍNDICE POR MARCA + AÑO (Pre-filtrado inteligente) ==========
print("\n[2/5] Construyendo índice de marca-año para filtrado rápido...")
marca_anio_index = defaultdict(lambda: defaultdict(list))

for idx, row in df_bse.iterrows():
    marca = row['marca_norm']
    try:
        anio_ini = int(row['anio_inicio'])
        anio_fin = int(row['anio_fin'])
        for year in range(anio_ini, anio_fin + 1):
            marca_anio_index[marca][year].append(idx)
    except (ValueError, TypeError):
        continue

print(f"Índice construido: {len(marca_anio_index)} marcas")

# ========== 3. EMBEDDINGS BSE (Búsqueda vectorial) ==========
print("\n[3/5] Generando embeddings BGE-M3 para BSE...")
bse_textos = (df_bse['marca_norm'] + " " + df_bse['modelo_norm']).tolist()

bse_embeddings = model_bi.encode(
    bse_textos,
    convert_to_tensor=False,
    batch_size=512,
    show_progress_bar=True,
    normalize_embeddings=True
).astype('float32')

print(f"Embeddings generados: {bse_embeddings.shape[0]} vectores ({bse_embeddings.shape[1]}D)")

# ========== 4. ÍNDICES BM25 POR MARCA ==========
print("\n[4/5] Construyendo índices BM25 por marca...")
bm25_cache = {}
marca_to_indices = defaultdict(list)

for idx, row in df_bse.iterrows():
    marca_to_indices[row['marca_norm']].append(idx)

for marca, indices in tqdm(marca_to_indices.items(), desc="BM25 por marca"):
    corpus_marca = [str(df_bse.loc[i, 'modelo_norm']).split() for i in indices]
    bm25_cache[marca] = {
        'bm25': BM25Okapi(corpus_marca),
        'indices': indices
    }

print(f"{len(bm25_cache)} índices BM25 cacheados")

# ========== 5. PRE-ENCODING AUTODATA + CALIBRACIÓN SIGMOID ==========
print("\n[5/5] Pre-encoding de queries Autodata y calibración de temperatura...")

# 5a. Batch encoding de todos los textos Autodata
# Evita encodear 1 query a la vez dentro del loop de matching (cuello de botella en GPU)
autodata_textos = (df_autodata['marca_norm'] + " " + df_autodata['modelo_norm']).tolist()
autodata_embeddings = model_bi.encode(
    autodata_textos,
    convert_to_tensor=False,
    batch_size=512,
    show_progress_bar=True,
    normalize_embeddings=True
).astype('float32')
print(f"Autodata embeddings pre-calculados: {autodata_embeddings.shape[0]} vectores ({autodata_embeddings.shape[1]}D)")

# 5b. Calibración automática de temperatura del sigmoid
# Lógica: muestreamos pares (query Autodata → top-1 BM25 por marca) y observamos
# la distribución de logits del reranker. Luego fijamos T tal que:
#   sigmoid(p80_logit / T) ≈ 0.95  →  T = p80 / ln(19) ≈ p80 / 2.944
# Esto garantiza que el 80% de los pares "candidatos" caigan en zona útil (> 0.5).
CALIBRACION_N = 150
pares_calibracion = []

# Para año conocido se verifica que exista al menos un candidato BSE compatible
# en marca y año, garantizando que cada registro de la muestra produzca un par válido.
# Para año desconocido (9999) alcanza con que la marca exista en BSE.
def _tiene_candidatos_calibracion(row):
    marcas = resolver_marcas_bse(row['marca_norm'])
    if row['anio'] != 9999:
        return any(
            mb in marca_anio_index and row['anio'] in marca_anio_index[mb]
            for mb in marcas
        )
    return any(mb in marca_to_indices for mb in marcas)

autodata_muestra = df_autodata[df_autodata.apply(_tiene_candidatos_calibracion, axis=1)]
if len(autodata_muestra) == 0:
    autodata_muestra = pd.DataFrame()
else:
    autodata_muestra = autodata_muestra.sample(min(CALIBRACION_N * 2, len(autodata_muestra)), random_state=42).reset_index(drop=True)

for _, row in autodata_muestra.iterrows():
    if len(pares_calibracion) >= CALIBRACION_N:
        break
    marca_auto = row['marca_norm']
    modelo_auto = row['modelo_norm']
    anio_auto  = row['anio']
    query_text = f"{marca_auto} {modelo_auto}"
    for m in resolver_marcas_bse(marca_auto):
        if m not in bm25_cache:
            continue
        all_indices = bm25_cache[m]['indices']
        # Respetar el filtro duro de año: solo candidatos cuyo rango incluye anio_auto
        if anio_auto != 9999:
            valid_set = set(marca_anio_index[m].get(anio_auto, []))
            positions = [pos for pos, idx in enumerate(all_indices) if idx in valid_set]
        else:
            positions = list(range(len(all_indices)))
        if not positions:
            continue
        bm25_scores = bm25_cache[m]['bm25'].get_scores(modelo_auto.split())
        best_pos    = max(positions, key=lambda p: bm25_scores[p])
        best_idx    = all_indices[best_pos]
        bse_text = (
            f"{df_bse.loc[best_idx, 'marca_norm']} "
            f"{df_bse.loc[best_idx, 'modelo_norm']} "
            f"{int(df_bse.loc[best_idx, 'anio_inicio'])}-{int(df_bse.loc[best_idx, 'anio_fin'])}"
        )
        pares_calibracion.append([
            f"{query_text} {anio_auto if anio_auto != 9999 else ''}",
            bse_text
        ])
        break

if pares_calibracion:
    logits_cal = reranker.predict(pares_calibracion)
    p10 = float(np.percentile(logits_cal, 10))
    p50 = float(np.percentile(logits_cal, 50))
    p80 = float(np.percentile(logits_cal, 80))
    # Temperatura acotada a [0.3, 2.0] para evitar extremos
    SIGMOID_TEMPERATURA = round(max(0.3, min(p80 / 2.944, 2.0)), 3) if p80 > 0.1 else 1.0
    print(f"\nCalibración sobre {len(pares_calibracion)} pares:")
    print(f"  Logits reranker — p10: {p10:.2f} | p50: {p50:.2f} | p80: {p80:.2f}")
    print(f"  Temperatura sigmoid calibrada automáticamente: {SIGMOID_TEMPERATURA}")
else:
    SIGMOID_TEMPERATURA = 1.0
    print("Calibración no disponible. Temperatura por defecto: 1.0")

print("\n" + "="*70)
print("SISTEMA OPTIMIZADO LISTO")
print("="*70)

## 7️⃣ Función de Matching Optimizada

In [ ]:

def calcular_penalizacion_anio(anio_query):
    """
    Penalización para registros Autodata con año desconocido (9999).
    Para años conocidos, el filtro duro de la Fase 1 garantiza que el año
    cae dentro del rango BSE, por lo que no se aplica penalización adicional.
    Retorna 0.9 para año desconocido y 1.0 para el resto.
    """
    if anio_query == 9999:  # Año desconocido: penalización leve
        return 0.9
    return 1.0


def sigmoid_temperatura(x, temperatura=1.0):
    """Convierte logits del reranker a [0,1] con escala calibrada para comparabilidad entre registros."""
    return 1.0 / (1.0 + np.exp(-x / temperatura))


def encontrar_matches_optimizado(row_autodata, top_k=3, top_candidates=50, query_embedding=None):
    """
    Pipeline de matching en 3 fases:
    1. Pre-filtrado por marca + año exacto: solo candidatos BSE cuyo rango incluye
       exactamente el año de Autodata (anio_inicio_bse <= ANIO_AUTODATA <= anio_fin_bse).
       Para año desconocido (9999) se usa el universo completo de la marca.
    2. Búsqueda híbrida: similitud semántica (BGE-M3) + léxica (BM25), restringida
       al mismo pool de candidatos filtrados por año.
    3. Reranking con cross-encoder. Para años desconocidos se aplica una penalización
       leve (0.9) sobre el score final.

    El score resultante (score_comparable) es globalmente consistente entre registros,
    lo que permite comparar la confianza de distintos matches entre sí.
    """
    marca_auto  = row_autodata['marca_norm']
    anio_auto   = row_autodata['anio']
    modelo_auto = row_autodata['modelo_norm']
    query_text  = f"{marca_auto} {modelo_auto}"

    # ========== FASE 1: PRE-FILTRADO POR MARCA Y AÑO EXACTO ==========
    marcas_bse = resolver_marcas_bse(marca_auto)

    # Se usa set comprehension para deduplicar índices cuando una marca tiene múltiples
    # alias en BSE y varios de ellos apuntan al mismo registro
    indices_marca = list({
        idx
        for m in marcas_bse
        if m in marca_to_indices
        for idx in marca_to_indices[m]
    })

    if not indices_marca:
        return []

    if anio_auto != 9999:
        # Filtro duro: solo candidatos BSE cuyo rango incluye exactamente el año de Autodata.
        # El índice marca_anio_index[marca][year] ya contiene exactamente los registros BSE
        # cuyo rango (anio_inicio, anio_fin) incluye ese year, por construcción.
        candidatos_set = set()
        for m in marcas_bse:
            if m in marca_anio_index and anio_auto in marca_anio_index[m]:
                candidatos_set.update(marca_anio_index[m][anio_auto])
        indices_candidatos = list(candidatos_set)
    else:
        # Año desconocido: no es posible filtrar por rango temporal
        indices_candidatos = indices_marca

    if not indices_candidatos:
        return []

    # ========== FASE 2: BÚSQUEDA HÍBRIDA ==========
    if query_embedding is not None:
        q_emb = query_embedding.reshape(1, -1).astype('float32')
    else:
        q_emb = model_bi.encode([query_text], normalize_embeddings=True).astype('float32')

    candidate_indices    = np.array(indices_candidatos)
    candidate_embeddings = bse_embeddings[candidate_indices]
    similarities         = np.dot(candidate_embeddings, q_emb.T).flatten()

    # Se toma el doble de top_candidates en semántica para que BM25 tenga suficiente pool
    # donde compensar casos donde el sentido semántico no es suficiente por sí solo
    top_sem_count     = min(top_candidates * 2, len(candidate_indices))
    top_sem_positions = np.argsort(similarities)[::-1][:top_sem_count]
    semantic_results  = [(int(candidate_indices[pos]), float(similarities[pos])) for pos in top_sem_positions]

    if not semantic_results:
        return []

    # BM25 léxico: complementa la búsqueda semántica para términos técnicos exactos
    # (versiones, cilindradas, abreviaturas) donde la similitud vectorial puede fallar.
    # Los scores se calculan sobre todo el corpus de la marca, pero solo se almacenan
    # los correspondientes a candidatos que pasaron el filtro de año (candidate_set).
    candidate_set   = set(indices_candidatos)
    bm25_dict       = {}
    tokenized_query = modelo_auto.split()
    for m in marcas_bse:
        if m in bm25_cache:
            bm25_data            = bm25_cache[m]
            bm25_model           = bm25_data['bm25']
            bm25_indices_locales = bm25_data['indices']
            bm25_scores          = bm25_model.get_scores(tokenized_query)
            if bm25_scores.max() > 0:
                bm25_scores = bm25_scores / bm25_scores.max()  # Normalización al rango [0, 1]
            for idx, score in zip(bm25_indices_locales, bm25_scores):
                if idx in candidate_set and (idx not in bm25_dict or score > bm25_dict[idx]):
                    bm25_dict[idx] = score

    # Combinación ponderada: más peso a semántica (70%) que a léxico (30%).
    combined_scores = []
    for idx, sem_score in semantic_results:
        lex_score    = bm25_dict.get(idx, 0)
        hybrid_score = 0.70 * sem_score + 0.30 * lex_score
        combined_scores.append((idx, hybrid_score))

    combined_scores.sort(key=lambda x: x[1], reverse=True)
    top_hybrid = combined_scores[:top_candidates]

    if not top_hybrid:
        return []

    # ========== FASE 3: RERANKING + SCORE COMPARABLE ==========
    indices_finales   = [idx for idx, _ in top_hybrid]
    hybrid_scores_raw = np.array([score for _, score in top_hybrid], dtype=float)

    # El texto del par incluye el año de Autodata para que el reranker
    # pueda considerarlo al comparar contra el rango de años BSE
    pairs = [
        [
            f"{query_text} {anio_auto if anio_auto != 9999 else ''}",
            f"{df_bse.loc[idx, 'marca_norm']} {df_bse.loc[idx, 'modelo_norm']} "
            f"{int(df_bse.loc[idx, 'anio_inicio'])}-{int(df_bse.loc[idx, 'anio_fin'])}"
        ]
        for idx in indices_finales
    ]

    rerank_scores = reranker.predict(pairs)

    # score_comparable combina reranker (85%) y score híbrido (15%).
    # La sigmoid con temperatura calibrada convierte los logits del reranker a [0,1]
    # de forma consistente entre registros.
    rerank_global    = sigmoid_temperatura(rerank_scores, temperatura=SIGMOID_TEMPERATURA)
    hybrid_global    = np.clip(hybrid_scores_raw, 0, 1)
    score_comparable = 0.85 * rerank_global + 0.15 * hybrid_global

    # Para año desconocido (9999) se aplica penalización leve (0.9); para el resto, 1.0.
    # Como el valor es escalar (no depende del candidato), se multiplica directamente.
    year_penalty = calcular_penalizacion_anio(anio_auto)
    score_comparable = score_comparable * year_penalty

    # Ordenar por score_comparable garantiza que el top_k retornado sea coherente
    # y directamente comparable entre distintas queries
    top_k_indices = np.argsort(score_comparable)[::-1][:top_k]

    return [{
        'id_bse':           df_bse.loc[indices_finales[i], 'ID_BSE'],
        'score_comparable': round(float(score_comparable[i]), 4),
        'marca_bse':        df_bse.loc[indices_finales[i], 'MARCA_BSE'],
        'modelo_bse':       df_bse.loc[indices_finales[i], 'MODELO_BSE'],
        'rango_anios_bse':  df_bse.loc[indices_finales[i], 'ANIOS_BSE'],
        'combustible_bse':  df_bse.loc[indices_finales[i], 'COMBUSTIBLE_BSE'],
        'tipo_bse':         df_bse.loc[indices_finales[i], 'TIPO_BSE'],
    } for i in top_k_indices]

print("Función de matching optimizada definida")


## 8️⃣ Ejecutar matching para todo el catálogo Autodata

In [ ]:

TOP_K = 3
resultados_finales = []

print(f"Iniciando matching de {len(df_autodata):,} registros...")

for enum_idx, (_, row) in enumerate(tqdm(df_autodata.iterrows(), total=len(df_autodata), desc="Matching")):
    matches = encontrar_matches_optimizado(
        row,
        top_k=TOP_K,
        query_embedding=autodata_embeddings[enum_idx]
    )

    base = {
        'ID_AUTODATA':     row['ID_AUTODATA'],
        'MARCA_AUTODATA':  row['MARCA_AUTODATA'],
        'MODELO_AUTODATA': row['MODELO_AUTODATA'],
        'ANIO_AUTODATA':   row['ANIO_AUTODATA'],
    }

    if not matches:
        # Sin candidatos: una sola fila centinela con ID_BSE = 'NO_MATCH'
        resultados_finales.append({
            **base,
            'RANK':             1,
            'SCORE_COMPARABLE': 0.0,
            'ID_BSE':           'NO_MATCH',
            'MARCA_BSE':        '',
            'MODELO_BSE':       '',
            'RANGO_ANIOS_BSE':  '',
            'COMBUSTIBLE_BSE':  '',
            'TIPO_BSE':         '',
        })
    else:
        for rank, m in enumerate(matches[:TOP_K], start=1):
            resultados_finales.append({
                **base,
                'RANK':             rank,
                # Todos los scores son score_comparable: globalmente consistentes entre registros
                'SCORE_COMPARABLE': m['score_comparable'],
                'ID_BSE':           m['id_bse'],
                'MARCA_BSE':        m['marca_bse'],
                'MODELO_BSE':       m['modelo_bse'],
                'RANGO_ANIOS_BSE':  m['rango_anios_bse'],
                'COMBUSTIBLE_BSE':  m['combustible_bse'],
                'TIPO_BSE':         m['tipo_bse'],
            })

df_resultados = pd.DataFrame(resultados_finales)

# Para estadísticas de cobertura se trabaja con RANK == 1 (un registro por entrada Autodata)
df_rank1 = df_resultados[df_resultados['RANK'] == 1]
n_sin_match = (df_rank1['ID_BSE'] == 'NO_MATCH').sum()
n_con_match = (df_rank1['ID_BSE'] != 'NO_MATCH').sum()

print("\n" + "="*70)
print("MATCHING COMPLETADO")
print("="*70)
print(f"Total registros Autodata procesados: {len(df_rank1):,}")
print(f"Sin matches:     {n_sin_match:,}")
print(f"Con matches:     {n_con_match:,}")
print(f"Total filas en CSV (top-{TOP_K} por registro): {len(df_resultados):,}")

df_con_match = df_rank1[df_rank1['ID_BSE'] != 'NO_MATCH']
if len(df_con_match) > 0:
    print(f"\nScore comparable promedio (mejor match): {df_con_match['SCORE_COMPARABLE'].mean():.3f}")

display(df_resultados.head(12))


## 9️⃣ Análisis de Calidad y Exportación

In [ ]:

print("\nDISTRIBUCIÓN DE SCORE_COMPARABLE (GLOBAL):\n")

# Las estadísticas se calculan sobre RANK == 1 (un registro por entrada Autodata).
# El criterio de exclusión es ID_BSE == 'NO_MATCH', no SCORE_COMPARABLE == 0,
# porque matches de baja confianza también tienen scores bajos pero son válidos.
df_rank1 = df_resultados[df_resultados['RANK'] == 1].copy()
df_con_match_global = df_rank1[df_rank1['ID_BSE'] != 'NO_MATCH'].copy()

if len(df_con_match_global) == 0:
    print("No se encontraron matches en el dataset. No hay estadísticas de score disponibles.")
    print(f"\nSin match (NO_MATCH): {len(df_rank1):,}")
else:
    p25 = df_con_match_global['SCORE_COMPARABLE'].quantile(0.25)
    p50 = df_con_match_global['SCORE_COMPARABLE'].quantile(0.50)
    p75 = df_con_match_global['SCORE_COMPARABLE'].quantile(0.75)

    print(f"Umbrales calculados sobre {len(df_con_match_global):,} registros con match:")
    print(f"  p25 = {p25:.4f} | p50 = {p50:.4f} | p75 = {p75:.4f}")
    print(f"  Temperatura sigmoid usada: {SIGMOID_TEMPERATURA}\n")

    print(f"Score >= {p75:.3f} (Alta confianza,    top 25%):     {(df_con_match_global['SCORE_COMPARABLE'] >= p75).sum():,}")
    print(f"Score {p50:.3f}-{p75:.3f} (Media confianza, 50-75%):  {((df_con_match_global['SCORE_COMPARABLE'] >= p50) & (df_con_match_global['SCORE_COMPARABLE'] < p75)).sum():,}")
    print(f"Score {p25:.3f}-{p50:.3f} (Revisión humana, 25-50%): {((df_con_match_global['SCORE_COMPARABLE'] >= p25) & (df_con_match_global['SCORE_COMPARABLE'] < p50)).sum():,}")
    print(f"Score <  {p25:.3f} (Baja confianza,   bot 25%):     {(df_con_match_global['SCORE_COMPARABLE'] < p25).sum():,}")
    print(f"\nSin match (NO_MATCH): {(df_rank1['ID_BSE'] == 'NO_MATCH').sum():,}")

archivo_salida = os.path.join(DRIVE_OUTPUT_PATH, 'resultados_matching_completo_v6.csv')
df_resultados.to_csv(archivo_salida, index=False, encoding='utf-8-sig')
print(f"\nResultados exportados a: {archivo_salida}")

print("\n" + "="*70)
print("========= PROCESO COMPLETADO EXITOSAMENTE =========")
print("="*70)


In [ ]:
# El archivo resumido conserva únicamente el mejor candidato por registro (RANK == 1)
df_filtrado = df_resultados[df_resultados['RANK'] == 1].copy()

columnas_resumido = [
    'ID_AUTODATA',
    'ID_BSE',
    'SCORE_COMPARABLE',
    'MARCA_AUTODATA',
    'MARCA_BSE',
    'ANIO_AUTODATA',
    'RANGO_ANIOS_BSE',
    'MODELO_AUTODATA',
    'MODELO_BSE',
    'COMBUSTIBLE_BSE',
    'TIPO_BSE',
]

df_filtrado = df_filtrado[columnas_resumido].copy()
print("DataFrame 'df_filtrado' creado con columnas finales estandarizadas.")
display(df_filtrado.head(10))

archivo_filtrado_salida = os.path.join(DRIVE_OUTPUT_PATH, "resultados_matching_resumido_v6.csv")
df_filtrado.to_csv(archivo_filtrado_salida, index=False, encoding="utf-8-sig")
print(f"DataFrame filtrado exportado a: {archivo_filtrado_salida}")


In [ ]:
df_bse.to_csv(os.path.join(DRIVE_OUTPUT_PATH, 'debug_df_bse.csv'), index=False, encoding='utf-8-sig')
df_autodata.to_csv(os.path.join(DRIVE_OUTPUT_PATH, 'debug_df_autodata.csv'), index=False, encoding='utf-8-sig')
df_validacion_ejemplos.to_csv(os.path.join(DRIVE_OUTPUT_PATH, 'debug_df_validacion_ejemplos.csv'), index=False, encoding='utf-8-sig')
df_revision_50.to_csv(os.path.join(DRIVE_OUTPUT_PATH, 'debug_df_revision_50.csv'), index=False, encoding='utf-8-sig')

print('4 archivos exportados.')
